# 03 风险因子分析

这个文件衔接 `01` 和 `02`：

- `01` 明确了风控目标和目标变量；
- `02` 处理了异常值、特殊编码和缺失标记；
- `03` 开始回答：哪些字段真的能区分坏客户风险？

本阶段不建模，只做最实用的风控分组坏客户率分析。


## 1. 本阶段目标

风险因子分析重点看三个指标：

1. 每个分组有多少客户；
2. 每个分组坏客户率是多少；
3. 分组坏客户率是否明显高于整体坏客户率。

如果一个分组坏客户率明显高于整体水平，它就是后续风险分层和策略规则的候选因子。


## 2. 导入工具和读取清洗后数据


In [37]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="Microsoft YaHei")

PROJECT_ROOT = Path(r"E:\DataAnalysis\DataAnalysisRoad\Projects\01_Credit_Risk_Analysis")
CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "credit_risk_cleaned.csv"
OUTPUT_TABLE_DIR = PROJECT_ROOT / "output" / "tables"
OUTPUT_FIGURE_DIR = PROJECT_ROOT / "output" / "figures"

OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_DATA_PATH


WindowsPath('E:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis/data/processed/credit_risk_cleaned.csv')

In [38]:
df = pd.read_csv(CLEAN_DATA_PATH)

df.head()


,customer_id,is_bad_customer,credit_utilization,age,past_due_30_59_count,debt_ratio,monthly_income,open_credit_count,past_due_90_count,real_estate_loan_count,past_due_60_89_count,dependent_count,is_monthly_income_missing,is_dependent_count_missing,is_age_zero,is_age_over_100,is_credit_utilization_over_1,is_open_credit_over_40,is_dependent_over_10,is_past_due_30_59_special,is_past_due_60_89_special,is_past_due_90_special,ever_dpd30_plus
0,1,1,0.7661,45.0000,2.0000,0.8030,"9,120.0000",13,0.0000,6,0.0000,2.0000,0,0,0,0,0,0,0,0,0,0,1.0000
1,2,0,0.9572,40.0000,0.0000,0.1219,"2,600.0000",4,0.0000,0,0.0000,1.0000,0,0,0,0,0,0,0,0,0,0,0.0000
2,3,0,0.6582,38.0000,1.0000,0.0851,"3,042.0000",2,1.0000,0,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,1.0000
3,4,0,0.2338,30.0000,0.0000,0.0360,"3,300.0000",5,0.0000,0,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,0.0000
4,5,0,0.9072,49.0000,1.0000,0.0249,"63,588.0000",7,0.0000,1,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0,1.0000


## 3. 整体风险基准

后面所有分组坏客户率都要和整体坏客户率比较。只有明显高于整体水平的分组，才有实际风控意义。


In [39]:
sample_count = len(df)
bad_count = df["is_bad_customer"].sum()
overall_bad_rate = df["is_bad_customer"].mean()

baseline_summary = pd.DataFrame({
    "指标": ["样本数", "坏客户数", "整体坏客户率"],
    "结果": [sample_count, bad_count, overall_bad_rate],
})

baseline_summary


,指标,结果
0,样本数,"150,000.0000"
1,坏客户数,"10,026.0000"
2,整体坏客户率,0.0668


## 4. 先看 02 生成的标记变量

这些变量来自数据清洗阶段。它们回答的是：缺失、异常、特殊编码本身是否有风险含义。


In [40]:
flag_summary = pd.DataFrame({
    "风险因子": [
        "月收入缺失",
        "家属数量缺失",
        "年龄原始值为0",
        "年龄大于100",
        "额度使用率大于1",
        "开户/贷款账户数大于40",
        "家属数量大于10",
        "30-59天逾期为特殊编码",
        "60-89天逾期为特殊编码",
        "90天以上逾期为特殊编码",
        "曾经出现30天及以上逾期",
    ],
    "字段": [
        "is_monthly_income_missing",
        "is_dependent_count_missing",
        "is_age_zero",
        "is_age_over_100",
        "is_credit_utilization_over_1",
        "is_open_credit_over_40",
        "is_dependent_over_10",
        "is_past_due_30_59_special",
        "is_past_due_60_89_special",
        "is_past_due_90_special",
        "ever_dpd30_plus",
    ],
})

flag_rows = []

for _, row in flag_summary.iterrows():
    field = row["字段"]
    mask = df[field].eq(1)
    flag_rows.append({
        "风险因子": row["风险因子"],
        "命中人数": mask.sum(),
        "命中占比": mask.mean(),
        "命中坏客户率": df.loc[mask, "is_bad_customer"].mean(),
        "相对整体坏客户率": df.loc[mask, "is_bad_customer"].mean() / overall_bad_rate,
    })

flag_risk_summary = pd.DataFrame(flag_rows).sort_values("相对整体坏客户率", ascending=False)
flag_risk_summary


,风险因子,命中人数,命中占比,命中坏客户率,相对整体坏客户率
7,30-59天逾期为特殊编码,269,0.0018,0.5465,8.1758
9,90天以上逾期为特殊编码,269,0.0018,0.5465,8.1758
8,60-89天逾期为特殊编码,269,0.0018,0.5465,8.1758
4,额度使用率大于1,3321,0.0221,0.3725,5.5727
10,曾经出现30天及以上逾期,30094,0.2006,0.2198,3.2886
5,开户/贷款账户数大于40,62,0.0004,0.1290,1.9305
3,年龄大于100,13,0.0001,0.0769,1.1509
0,月收入缺失,29731,0.1982,0.0561,0.8399
1,家属数量缺失,3924,0.0262,0.0456,0.6825
2,年龄原始值为0,1,0.0000,0.0000,0.0000


## 5. 逾期记录：最直接的风险信号

历史逾期是风控中最重要的风险因子之一。

这里要严格区分两类信息：

- 正式逾期风险分析：只使用 DPD 状态可判断的客户；
- 特殊编码观察：逾期字段为 96/98 的客户单独观察，不纳入正式逾期分组。

这样可以避免把“数据质量异常”误解释成“正常逾期等级”。


In [41]:
known_dpd = df["ever_dpd30_plus"].notna()
known_dpd_bad_rate = df.loc[known_dpd, "is_bad_customer"].mean()

# 正式 DPD30+ 分析：只看可以判断 DPD30+ 状态的客户。
dpd30_status = df.loc[known_dpd, "ever_dpd30_plus"].map({
    0.0: "无DPD30+",
    1.0: "有DPD30+",
})

dpd30_summary = df.loc[known_dpd].groupby(dpd30_status)["is_bad_customer"].agg(["count", "mean"]).reset_index()
dpd30_summary.columns = ["DPD30+状态", "客户数", "坏客户率"]
dpd30_summary["客户占比"] = dpd30_summary["客户数"] / known_dpd.sum()
dpd30_summary["相对可判断DPD客群坏客户率"] = dpd30_summary["坏客户率"] / known_dpd_bad_rate
dpd30_summary["相对整体坏客户率"] = dpd30_summary["坏客户率"] / overall_bad_rate

dpd30_summary


,DPD30+状态,客户数,坏客户率,客户占比,相对可判断DPD客群坏客户率,相对整体坏客户率
0,无DPD30+,119637,0.0273,0.7990,0.4135,0.4082
1,有DPD30+,30094,0.2198,0.2010,3.3316,3.2886


In [42]:
# 90天以上逾期次数分析：同样只使用可判断逾期状态的客户。
past_due_90_group = pd.cut(
    df.loc[known_dpd, "past_due_90_count"],
    bins=[-0.1, 0, 1, 2, 100],
    labels=["0次", "1次", "2次", "3次及以上"],
)

past_due_90_summary = df.loc[known_dpd].groupby(past_due_90_group)["is_bad_customer"].agg(["count", "mean"]).reset_index()
past_due_90_summary.columns = ["90天以上逾期次数", "客户数", "坏客户率"]
past_due_90_summary["客户占比"] = past_due_90_summary["客户数"] / known_dpd.sum()
past_due_90_summary["相对可判断DPD客群坏客户率"] = past_due_90_summary["坏客户率"] / known_dpd_bad_rate
past_due_90_summary["相对整体坏客户率"] = past_due_90_summary["坏客户率"] / overall_bad_rate

past_due_90_summary


,90天以上逾期次数,客户数,坏客户率,客户占比,相对可判断DPD客群坏客户率,相对整体坏客户率
0,0次,141662,0.0463,0.9461,0.7012,0.6922
1,1次,5243,0.3366,0.0350,5.1023,5.0365
2,2次,1555,0.4990,0.0104,7.5636,7.4661
3,3次及以上,1271,0.6168,0.0085,9.3491,9.2286


## 6. 年龄分组

年龄通常不是单调风险变量，不能只看平均值。实际工作中更常用分组坏客户率。


In [43]:
age_group = pd.cut(
    df["age"],
    bins=[0, 25, 35, 45, 55, 65, 75, 120],
    labels=["25岁及以下", "26-35岁", "36-45岁", "46-55岁", "56-65岁", "66-75岁", "75岁以上"],
)

age_summary = df.groupby(age_group)["is_bad_customer"].agg(["count", "mean"]).reset_index()
age_summary.columns = ["年龄分组", "客户数", "坏客户率"]
age_summary["客户占比"] = age_summary["客户数"] / sample_count
age_summary["相对整体坏客户率"] = age_summary["坏客户率"] / overall_bad_rate

age_summary


,年龄分组,客户数,坏客户率,客户占比,相对整体坏客户率
0,25岁及以下,3027,0.1117,0.0202,1.6706
1,26-35岁,18458,0.1112,0.1231,1.6641
2,36-45岁,29819,0.0881,0.1988,1.3185
3,46-55岁,36690,0.0759,0.2446,1.1360
4,56-65岁,33406,0.0458,0.2227,0.6857
5,66-75岁,18470,0.0262,0.1231,0.3912
6,75岁以上,10129,0.0204,0.0675,0.3058


In [44]:
plt.figure(figsize=(8, 4))
sns.barplot(data=age_summary, x="年龄分组", y="坏客户率", color="#4C78A8")
plt.axhline(overall_bad_rate, color="red", linestyle="--", label="整体坏客户率")
plt.title("不同年龄分组的坏客户率")
plt.xticks(rotation=30)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FIGURE_DIR / "age_bad_rate.png", dpi=150)
plt.close()


## 7. 额度使用率分组

额度使用率越高，通常表示客户资金压力越大。这里重点观察使用率超过 1 的客户风险。


In [45]:
util_group = pd.cut(
    df["credit_utilization"],
    bins=[-0.1, 0, 0.3, 0.7, 1, 10, df["credit_utilization"].max()],
    labels=["0", "0-30%", "30%-70%", "70%-100%", "100%-1000%", "1000%以上"],
)

util_summary = df.groupby(util_group)["is_bad_customer"].agg(["count", "mean"]).reset_index()
util_summary.columns = ["额度使用率分组", "客户数", "坏客户率"]
util_summary["客户占比"] = util_summary["客户数"] / sample_count
util_summary["相对整体坏客户率"] = util_summary["坏客户率"] / overall_bad_rate

util_summary


,额度使用率分组,客户数,坏客户率,客户占比,相对整体坏客户率
0,0,10878,0.0294,0.0725,0.4401
1,0-30%,82004,0.0212,0.5467,0.3176
2,30%-70%,27170,0.0740,0.1811,1.1074
3,70%-100%,26627,0.1772,0.1775,2.6504
4,100%-1000%,3080,0.3961,0.0205,5.9262
5,1000%以上,241,0.0705,0.0016,1.0553


## 8. 收入分组

收入缺失不能直接删除。这里把缺失单独作为一个分组，观察它是否有风险含义。


In [46]:
income_group = pd.cut(
    df["monthly_income"],
    bins=[-0.1, 2000, 5000, 10000, 20000, df["monthly_income"].max()],
    labels=["0-2000", "2000-5000", "5000-10000", "10000-20000", "20000以上"],
)

income_group = income_group.astype("object")
income_group[df["monthly_income"].isna()] = "收入缺失"

income_summary = df.groupby(income_group)["is_bad_customer"].agg(["count", "mean"]).reset_index()
income_summary.columns = ["月收入分组", "客户数", "坏客户率"]
income_summary["客户占比"] = income_summary["客户数"] / sample_count
income_summary["相对整体坏客户率"] = income_summary["坏客户率"] / overall_bad_rate

income_summary


,月收入分组,客户数,坏客户率,客户占比,相对整体坏客户率
0,0-2000,12003,0.0845,0.0800,1.2639
1,10000-20000,16216,0.0419,0.1081,0.6274
2,2000-5000,43856,0.0866,0.2924,1.2960
3,20000以上,2103,0.0533,0.0140,0.7968
4,5000-10000,46091,0.0597,0.3073,0.8933
5,收入缺失,29731,0.0561,0.1982,0.8399


## 9. 负债率分组

负债率在这个数据中有明显极端值。这里先做粗分组，不在本阶段解释每个极端值的成因。


In [47]:
debt_group = pd.cut(
    df["debt_ratio"],
    bins=[-0.1, 0.2, 0.5, 1, 5, df["debt_ratio"].max()],
    labels=["0-20%", "20%-50%", "50%-100%", "100%-500%", "500%以上"],
)

debt_summary = df.groupby(debt_group)["is_bad_customer"].agg(["count", "mean"]).reset_index()
debt_summary.columns = ["负债率分组", "客户数", "坏客户率"]
debt_summary["客户占比"] = debt_summary["客户数"] / sample_count
debt_summary["相对整体坏客户率"] = debt_summary["坏客户率"] / overall_bad_rate

debt_summary


,负债率分组,客户数,坏客户率,客户占比,相对整体坏客户率
0,0-20%,42289,0.0611,0.2819,0.9145
1,20%-50%,51419,0.0597,0.3428,0.8933
2,50%-100%,21155,0.0983,0.1410,1.4710
3,100%-500%,5491,0.1182,0.0366,1.7683
4,500%以上,29646,0.0554,0.1976,0.8286


## 10. 开放信贷账户数分组

账户数过少可能代表信用历史薄，账户数过多可能代表多头借贷。两端都需要观察。


In [48]:
open_credit_group = pd.cut(
    df["open_credit_count"],
    bins=[-0.1, 0, 3, 8, 15, 40, df["open_credit_count"].max()],
    labels=["0个", "1-3个", "4-8个", "9-15个", "16-40个", "40个以上"],
)

open_credit_summary = df.groupby(open_credit_group)["is_bad_customer"].agg(["count", "mean"]).reset_index()
open_credit_summary.columns = ["开放信贷账户数分组", "客户数", "坏客户率"]
open_credit_summary["客户占比"] = open_credit_summary["客户数"] / sample_count
open_credit_summary["相对整体坏客户率"] = open_credit_summary["坏客户率"] / overall_bad_rate

open_credit_summary


,开放信贷账户数分组,客户数,坏客户率,客户占比,相对整体坏客户率
0,0个,1888,0.2564,0.0126,3.8354
1,1-3个,20162,0.0930,0.1344,1.3921
2,4-8个,63961,0.0574,0.4264,0.8582
3,9-15个,50163,0.0606,0.3344,0.9073
4,16-40个,13764,0.0688,0.0918,1.0294
5,40个以上,62,0.1290,0.0004,1.9305


## 11. 家属数量分组

家属数量缺失和极端值都在 `02` 中做了标记。这里观察家庭负担和坏客户率的关系。


In [49]:
dependent_group = pd.cut(
    df["dependent_count"],
    bins=[-0.1, 0, 1, 2, 5, df["dependent_count"].max()],
    labels=["0人", "1人", "2人", "3-5人", "5人以上"],
)

dependent_group = dependent_group.astype("object")
dependent_group[df["dependent_count"].isna()] = "家属数量缺失"

dependent_summary = df.groupby(dependent_group)["is_bad_customer"].agg(["count", "mean"]).reset_index()
dependent_summary.columns = ["家属数量分组", "客户数", "坏客户率"]
dependent_summary["客户占比"] = dependent_summary["客户数"] / sample_count
dependent_summary["相对整体坏客户率"] = dependent_summary["坏客户率"] / overall_bad_rate

dependent_summary


,家属数量分组,客户数,坏客户率,客户占比,相对整体坏客户率
0,0人,86902,0.0586,0.5793,0.8772
1,1人,26316,0.0735,0.1754,1.1001
2,2人,19522,0.0811,0.1301,1.2139
3,3-5人,13091,0.0918,0.0873,1.3737
4,5人以上,245,0.1265,0.0016,1.8930
5,家属数量缺失,3924,0.0456,0.0262,0.6825


## 12. 汇总高风险分组

这张表用于项目报告：把坏客户率高于整体水平 1.5 倍的正式风险分组列出来。

注意：逾期字段特殊编码属于数据质量标记，只保留在 `flag_risk_summary` 中观察，不进入正式高风险分组。


In [ ]:
dpd30_report = dpd30_summary.rename(columns={"DPD30+状态": "分组"})
dpd30_report["风险因子"] = "DPD30+状态"

past_due_90_report = past_due_90_summary.rename(columns={"90天以上逾期次数": "分组"})
past_due_90_report["风险因子"] = "90天以上逾期次数"

age_report = age_summary.rename(columns={"年龄分组": "分组"})
age_report["风险因子"] = "年龄"

util_report = util_summary.rename(columns={"额度使用率分组": "分组"})
util_report["风险因子"] = "额度使用率"

income_report = income_summary.rename(columns={"月收入分组": "分组"})
income_report["风险因子"] = "月收入"

debt_report = debt_summary.rename(columns={"负债率分组": "分组"})
debt_report["风险因子"] = "负债率"

open_credit_report = open_credit_summary.rename(columns={"开放信贷账户数分组": "分组"})
open_credit_report["风险因子"] = "开放信贷账户数"

dependent_report = dependent_summary.rename(columns={"家属数量分组": "分组"})
dependent_report["风险因子"] = "家属数量"

risk_factor_summary = pd.concat(
    [
        dpd30_report,
        past_due_90_report,
        age_report,
        util_report,
        income_report,
        debt_report,
        open_credit_report,
        dependent_report,
    ],
    ignore_index=True,
)

risk_factor_summary = risk_factor_summary[["风险因子", "分组", "客户数", "客户占比", "坏客户率", "相对整体坏客户率"]]
high_risk_groups = risk_factor_summary[risk_factor_summary["相对整体坏客户率"] >= 1.5].sort_values(
    "相对整体坏客户率",
    ascending=False,
)

risk_factor_summary.head(),high_risk_groups


(        风险因子       分组     客户数   客户占比   坏客户率  相对整体坏客户率
 0   DPD30+状态  无DPD30+  119637 0.7990 0.0273    0.4082
 1   DPD30+状态  有DPD30+   30094 0.2010 0.2198    3.2886
 2  90天以上逾期次数       0次  141662 0.9461 0.0463    0.6922
 3  90天以上逾期次数       1次    5243 0.0350 0.3366    5.0365
 4  90天以上逾期次数       2次    1555 0.0104 0.4990    7.4661,
          风险因子          分组    客户数   客户占比   坏客户率  相对整体坏客户率
 5   90天以上逾期次数       3次及以上   1271 0.0085 0.6168    9.2286
 4   90天以上逾期次数          2次   1555 0.0104 0.4990    7.4661
 17      额度使用率  100%-1000%   3080 0.0205 0.3961    5.9262
 3   90天以上逾期次数          1次   5243 0.0350 0.3366    5.0365
 30    开放信贷账户数          0个   1888 0.0126 0.2564    3.8354
 1    DPD30+状态     有DPD30+  30094 0.2010 0.2198    3.2886
 16      额度使用率    70%-100%  26627 0.1775 0.1772    2.6504
 35    开放信贷账户数       40个以上     62 0.0004 0.1290    1.9305
 40       家属数量        5人以上    245 0.0016 0.1265    1.8930
 28        负债率   100%-500%   5491 0.0366 0.1182    1.7683
 6          年龄      25岁及以下   30

: 

## 13. 保存分析结果


In [51]:
risk_factor_summary_path = OUTPUT_TABLE_DIR / "risk_factor_summary.csv"
high_risk_groups_path = OUTPUT_TABLE_DIR / "high_risk_groups.csv"
flag_risk_summary_path = OUTPUT_TABLE_DIR / "flag_risk_summary.csv"

risk_factor_summary.to_csv(risk_factor_summary_path, index=False, encoding="utf-8-sig")
high_risk_groups.to_csv(high_risk_groups_path, index=False, encoding="utf-8-sig")
flag_risk_summary.to_csv(flag_risk_summary_path, index=False, encoding="utf-8-sig")

risk_factor_summary_path, high_risk_groups_path, flag_risk_summary_path


(WindowsPath('E:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis/output/tables/risk_factor_summary.csv'),
 WindowsPath('E:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis/output/tables/high_risk_groups.csv'),
 WindowsPath('E:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis/output/tables/flag_risk_summary.csv'))

## 14. 本阶段结论

- 历史逾期相关变量是最直接的风险信号；
- 逾期字段特殊编码不作为正式逾期等级，只作为数据质量标记单独观察；
- 缺失值和异常标记不能简单删除，因为它们可能本身有风险含义；
- 年龄、额度使用率、收入、负债率、账户数、家属数量需要通过分组坏客户率判断风险方向；
- 下一步可以进入 `04_risk_segmentation.ipynb`，把高风险分组组合成简单的风险分层规则。
